In [1]:
# manipulacja danymi
import numpy as np
import pandas as pd

# podzial danych
from sklearn.model_selection import train_test_split, GridSearchCV

# budowa Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures

# model
from sklearn.linear_model import LinearRegression

# ewaluacja regresji
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
dataset = pd.read_csv('daily-bike-share.csv')

In [3]:
dataset.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,rentals
0,1,1/1/2011,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331
1,2,1/2/2011,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131
2,3,1/3/2011,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120
3,4,1/4/2011,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108
4,5,1/5/2011,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82


In [4]:
dataset.iloc[0]

instant              1
dteday        1/1/2011
season               1
yr                   0
mnth                 1
holiday              0
weekday              6
workingday           0
weathersit           2
temp          0.344167
atemp         0.363625
hum           0.805833
windspeed     0.160446
rentals            331
Name: 0, dtype: object

In [5]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    object 
 2   season      731 non-null    int64  
 3   yr          731 non-null    int64  
 4   mnth        731 non-null    int64  
 5   holiday     731 non-null    int64  
 6   weekday     731 non-null    int64  
 7   workingday  731 non-null    int64  
 8   weathersit  731 non-null    int64  
 9   temp        731 non-null    float64
 10  atemp       731 non-null    float64
 11  hum         731 non-null    float64
 12  windspeed   731 non-null    float64
 13  rentals     731 non-null    int64  
dtypes: float64(4), int64(9), object(1)
memory usage: 80.1+ KB


In [6]:
X = dataset.drop(['rentals', 'instant'], axis=1)
y = dataset['rentals']


In [7]:
# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

In [8]:
# =========================
# PODZIAL KOLUMN
# =========================

num_features = [
    'temp',
    'atemp',
    'hum',
    'windspeed'
]

cat_features = [
    'season',
    'yr',
    'mnth',
    'holiday',
    'weekday',
    'workingday',
    'weathersit'
]

In [9]:
# =========================
# PRZYGOTOWANIE DANYCH
# =========================

num_preparation = Pipeline(steps=[
    ('scaler', StandardScaler())
])

cat_preparation = Pipeline(steps=[
    ('encoder', OneHotEncoder(
        drop='first',
        handle_unknown='ignore',
        sparse_output=False
    ))
])

data_preparation = ColumnTransformer(
    transformers=[
        ('num', num_preparation, num_features),
        ('cat', cat_preparation, cat_features)
    ],
    remainder='drop'
)


In [10]:
# =========================
# MODEL BAZOWY
# =========================

model_pipeline = Pipeline(steps=[
    ('preprocessor', data_preparation),
    ('model', LinearRegression())
])

model_pipeline.fit(X_train, y_train)

y_predict_train = model_pipeline.predict(X_train)
y_predict_test = model_pipeline.predict(X_test)


In [11]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'R2': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse)
    }

baseline_results = pd.DataFrame([
    {'zbior': 'train', **regression_metrics(y_train, y_predict_train)},
    {'zbior': 'test', **regression_metrics(y_test, y_predict_test)}
])

baseline_results


,zbior,R2,MAE,MSE,RMSE
0,train,0.747889,262.025212,127985.40112,357.750473
1,test,0.697731,238.871412,104446.63231,323.182042


In [12]:
# =========================
# EKSPERYMENT: POLYNOMIAL FEATURES + GRID SEARCH
# =========================

num_preparation_poly = Pipeline(steps=[
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler())
])

data_preparation_poly = ColumnTransformer(
    transformers=[
        ('num', num_preparation_poly, num_features),
        ('cat', cat_preparation, cat_features)
    ],
    remainder='drop'
)

poly_pipeline = Pipeline(steps=[
    ('preprocessor', data_preparation_poly),
    ('model', LinearRegression())
])

param_grid = {
    'preprocessor__num__poly__degree': [1, 2, 3]
}

grid_search = GridSearchCV(
    estimator=poly_pipeline,
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

cv_results = pd.DataFrame(grid_search.cv_results_)[[
    'param_preprocessor__num__poly__degree',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]].sort_values('rank_test_score')

cv_results


,param_preprocessor__num__poly__degree,mean_test_score,std_test_score,rank_test_score
1,2,0.712291,0.022885,1
0,1,0.707558,0.020912,2
2,3,0.692377,0.059792,3


In [13]:
best_poly_model = grid_search.best_estimator_

y_predict_poly_train = best_poly_model.predict(X_train)
y_predict_poly_test = best_poly_model.predict(X_test)

poly_degree = grid_search.best_params_['preprocessor__num__poly__degree']
poly_results = pd.DataFrame([
    {'model': 'bazowy', 'zbior': 'train', **regression_metrics(y_train, y_predict_train)},
    {'model': 'bazowy', 'zbior': 'test', **regression_metrics(y_test, y_predict_test)},
    {'model': f'PolynomialFeatures degree={poly_degree}', 'zbior': 'train', **regression_metrics(y_train, y_predict_poly_train)},
    {'model': f'PolynomialFeatures degree={poly_degree}', 'zbior': 'test', **regression_metrics(y_test, y_predict_poly_test)}
])

print('Najlepszy stopien PolynomialFeatures:', poly_degree)
print(f'Najlepszy sredni wynik CV R2: {grid_search.best_score_:.4f}')
poly_results


Najlepszy stopien PolynomialFeatures: 2
Najlepszy sredni wynik CV R2: 0.7123


,model,zbior,R2,MAE,MSE,RMSE
0,bazowy,train,0.747889,262.025212,127985.401120,357.750473
1,bazowy,test,0.697731,238.871412,104446.632310,323.182042
2,PolynomialFeatures degree=2,train,0.765257,254.165279,119168.452694,345.207840
3,PolynomialFeatures degree=2,test,-0.508415,283.562310,521220.248246,721.955849


# WYNIKI z MODULU 12

Metryka Model końcowy 0 R² 0.524682 1 MAE 348.585194 2 MSE 234195.931639 3 RMSE 483.937942

# WNIOSKI



W ramach zadania zbudowano Pipeline automatyzujący proces przygotowania danych oraz trenowania modelu regresji liniowej. Dla zmiennych numerycznych zastosowano standaryzację, natomiast zmienne kategoryczne zostały zakodowane przy użyciu One-Hot Encoding. Następnie wykorzystano model LinearRegression do przewidywania liczby wypożyczeń rowerów.

Przeprowadzono również eksperyment z wykorzystaniem PolynomialFeatures, jednak spowodował on przeuczenie modelu. Pomimo wysokiego wyniku na zbiorze treningowym, jakość predykcji na zbiorze testowym znacząco spadła, co świadczyło o słabej zdolności generalizacji.

Ostateczny Pipeline bez cech wielomianowych osiągnął współczynnik determinacji R² równy 0.6977 na zbiorze testowym, co stanowi znaczącą poprawę względem wcześniejszego modelu, który uzyskał wynik R² równy 0.5247. Dodatkowo zmniejszeniu uległy błędy MAE, MSE oraz RMSE, co potwierdza wyższą jakość predykcji. Zastosowanie Pipeline pozwoliło uporządkować proces przygotowania danych, ograniczyć ryzyko błędów oraz ułatwić dalszą rozbudowę modelu.
